In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data/processed', exist_ok=True)

# Check UCI heart disease columns
uci = pd.read_csv('../data/raw/uci_heart_disease.csv')
print(f"UCI shape: {uci.shape}")
print(f"UCI columns: {list(uci.columns)}")
print(uci.head(3).to_string())

UCI shape: (297, 14)
UCI columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'condition']
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  ca  thal  condition
0   69    1   0       160   234    1        2      131      0      0.1      1   1     0          0
1   69    0   0       140   239    0        0      151      0      1.8      0   2     0          0
2   66    0   0       150   226    0        0      114      0      2.6      2   0     0          0


In [2]:
hyp = pd.read_csv('../data/raw/hypertension_data.csv')
print(f"Hypertension shape: {hyp.shape}")
print(f"Hypertension columns: {list(hyp.columns)}")
print(hyp.head(3).to_string())

Hypertension shape: (68205, 17)
Hypertension columns: ['id', 'age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio', 'age_years', 'bmi', 'bp_category', 'bp_category_encoded']
   id    age  gender  height  weight  ap_hi  ap_lo  cholesterol  gluc  smoke  alco  active  cardio  age_years        bmi           bp_category   bp_category_encoded
0   0  18393       2     168    62.0    110     80            1     1      0     0       1       0         50  21.967120  Hypertension Stage 1  Hypertension Stage 1
1   1  20228       1     156    85.0    140     90            3     1      0     0       1       1         55  34.927679  Hypertension Stage 2  Hypertension Stage 2
2   2  18857       1     165    64.0    130     70            3     1      0     0       0       1         51  23.507805  Hypertension Stage 1  Hypertension Stage 1


In [3]:
synthetic = pd.read_csv('../data/synthetic/synthetic_data.csv')
print(f"✅ Synthetic loaded: {synthetic.shape}")

✅ Synthetic loaded: (50000, 22)


In [4]:
# Drop irrelevant columns
uci = uci.drop(columns=['sex', 'cp', 'chol', 'fbs', 'restecg', 'exang', 'oldpeak', 'slope', 'ca', 'thal'])

# Rename to standard names
uci = uci.rename(columns={
    'trestbps': 'avg_systolic',
    'thalach': 'avg_pulse',
    'condition': 'hypertension_label'
})

# Derive diastolic from systolic
uci['avg_diastolic'] = (uci['avg_systolic'] * 0.65 + np.random.normal(0, 5, len(uci))).clip(55, 120).round(1)
uci['bp_variability'] = np.random.normal(8, 3, len(uci)).clip(0, 25).round(1)
uci['avg_glucose'] = np.random.normal(95, 20, len(uci)).clip(60, 300).round(1)
uci['sugar_variability'] = np.random.normal(12, 4, len(uci)).clip(0, 40).round(1)
uci['bmi'] = np.random.normal(27, 5, len(uci)).clip(15, 50).round(1)
uci['adherence_rate'] = np.random.normal(0.80, 0.12, len(uci)).clip(0.1, 1.0).round(2)
uci['missed_doses'] = np.round((1 - uci['adherence_rate']) * 21).astype(int).clip(0, 15)
uci['has_chronic_condition'] = (uci['hypertension_label'] == 1).astype(int)
uci['allergies_count'] = np.random.choice([0,1,2,3], len(uci), p=[0.5,0.25,0.15,0.10])
uci['physical_activity_score'] = np.random.normal(5, 2, len(uci)).clip(0, 10).round(1)
uci['diet_quality_score'] = np.random.normal(5.5, 2, len(uci)).clip(0, 10).round(1)
uci['stress_score'] = np.random.normal(5, 2, len(uci)).clip(0, 10).round(1)
uci['sleep_score'] = np.random.normal(6, 1.5, len(uci)).clip(0, 10).round(1)

# Assign hypertension risk based on BP
def assign_hypertension_risk(row):
    if row['avg_systolic'] > 140 or row['avg_diastolic'] > 90:
        return 'high'
    elif row['avg_systolic'] > 130 or row['avg_diastolic'] > 80:
        return 'medium'
    else:
        return 'low'

uci['hypertension_risk'] = uci.apply(assign_hypertension_risk, axis=1)

print(f"✅ UCI cleaned: {uci.shape}")
print(f"\nHypertension Risk Distribution:")
print(uci['hypertension_risk'].value_counts())

✅ UCI cleaned: (297, 18)

Hypertension Risk Distribution:
hypertension_risk
low       112
high      103
medium     82
Name: count, dtype: int64


In [5]:
# Drop irrelevant columns
hyp = hyp.drop(columns=['id', 'age', 'gender', 'height', 'weight',
                          'cholesterol', 'gluc', 'smoke', 'alco',
                          'bp_category', 'bp_category_encoded'])

# Rename to standard names
hyp = hyp.rename(columns={
    'ap_hi': 'avg_systolic',
    'ap_lo': 'avg_diastolic',
    'age_years': 'age',
    'active': 'physical_activity_raw',
    'cardio': 'hypertension_label'
})

# Clean BP values (remove outliers)
hyp = hyp[
    (hyp['avg_systolic'] >= 80) & (hyp['avg_systolic'] <= 220) &
    (hyp['avg_diastolic'] >= 50) & (hyp['avg_diastolic'] <= 140)
]

# BP variability
hyp['bp_variability'] = np.random.normal(8, 3, len(hyp)).clip(0, 25).round(1)

# Glucose derived from cardio label
hyp['avg_glucose'] = (
    85 + (hyp['hypertension_label'] * 20) +
    np.random.normal(0, 15, len(hyp))
).clip(60, 300).round(1)

hyp['sugar_variability'] = np.random.normal(12, 4, len(hyp)).clip(0, 40).round(1)
hyp['avg_pulse'] = np.random.normal(75, 10, len(hyp)).clip(50, 110).round(1)

# Physical activity
hyp['physical_activity_score'] = (
    hyp['physical_activity_raw'] * 7 +
    np.random.normal(0, 1.5, len(hyp))
).clip(0, 10).round(1)

hyp['adherence_rate'] = (
    0.85 - (hyp['hypertension_label'] * 0.08) +
    np.random.normal(0, 0.1, len(hyp))
).clip(0.1, 1.0).round(2)

hyp['missed_doses'] = np.round(
    (1 - hyp['adherence_rate']) * 21
).astype(int).clip(0, 15)

hyp['has_chronic_condition'] = (hyp['hypertension_label'] == 1).astype(int)
hyp['allergies_count'] = np.random.choice([0,1,2,3], len(hyp), p=[0.5,0.25,0.15,0.10])
hyp['diet_quality_score'] = np.random.normal(5.5, 2, len(hyp)).clip(0, 10).round(1)
hyp['stress_score'] = np.random.normal(5, 2, len(hyp)).clip(0, 10).round(1)
hyp['sleep_score'] = np.random.normal(6, 1.5, len(hyp)).clip(0, 10).round(1)

# Assign hypertension risk
hyp['hypertension_risk'] = hyp.apply(assign_hypertension_risk, axis=1)

print(f"✅ Hypertension dataset cleaned: {hyp.shape}")
print(f"\nHypertension Risk Distribution:")
print(hyp['hypertension_risk'].value_counts())

✅ Hypertension dataset cleaned: (68205, 19)

Hypertension Risk Distribution:
hypertension_risk
low       44440
medium    13663
high      10102
Name: count, dtype: int64


In [6]:
hypertension_cols = [
    'age', 'bmi', 'avg_systolic', 'avg_diastolic',
    'bp_variability', 'avg_glucose', 'sugar_variability',
    'avg_pulse', 'adherence_rate', 'missed_doses',
    'has_chronic_condition', 'physical_activity_score',
    'diet_quality_score', 'stress_score', 'sleep_score',
    'hypertension_risk'
]

synthetic_hyp = synthetic[hypertension_cols].copy()
print(f"✅ Synthetic hypertension subset: {synthetic_hyp.shape}")

✅ Synthetic hypertension subset: (50000, 16)


In [7]:
# Select same columns
uci_final = uci[hypertension_cols].copy()
hyp_final = hyp[hypertension_cols].copy()

# Combine
combined = pd.concat([
    uci_final,
    hyp_final,
    synthetic_hyp
], ignore_index=True)

# Shuffle
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

# Fill nulls
combined = combined.fillna(combined.median(numeric_only=True))

print(f"✅ Combined dataset: {combined.shape}")
print(f"\nHypertension Risk Distribution:")
print(combined['hypertension_risk'].value_counts())
print(f"\nNull values: {combined.isnull().sum().sum()}")

✅ Combined dataset: (118502, 16)

Hypertension Risk Distribution:
hypertension_risk
low       61223
medium    31313
high      25966
Name: count, dtype: int64

Null values: 0


In [8]:
print("📊 Final Hypertension Dataset Summary:")
print(f"   Total rows: {len(combined):,}")
print(f"   Total columns: {len(combined.columns)}")
print(f"   Columns: {list(combined.columns)}")

print(f"\nFeature Statistics:")
print(combined.describe().round(2))

# Save
output_path = '../data/processed/hypertension_features.csv'
combined.to_csv(output_path, index=False)

print(f"\n✅ Hypertension features saved!")
print(f"   Path: {output_path}")
print(f"   Rows: {len(combined):,}")
print(f"   Size: {os.path.getsize(output_path)/1024/1024:.1f} MB")

📊 Final Hypertension Dataset Summary:
   Total rows: 118,502
   Total columns: 16
   Columns: ['age', 'bmi', 'avg_systolic', 'avg_diastolic', 'bp_variability', 'avg_glucose', 'sugar_variability', 'avg_pulse', 'adherence_rate', 'missed_doses', 'has_chronic_condition', 'physical_activity_score', 'diet_quality_score', 'stress_score', 'sleep_score', 'hypertension_risk']

Feature Statistics:
             age        bmi  avg_systolic  avg_diastolic  bp_variability  \
count  118502.00  118502.00     118502.00      118502.00       118502.00   
mean       49.41      26.41        128.09          81.32            9.15   
std        11.43       6.11         14.52           8.61            3.31   
min        18.00       3.47         85.00          55.00            0.00   
25%        43.00      22.86        120.00          77.70            7.20   
50%        51.00      25.71        126.20          80.00            9.00   
75%        57.00      29.60        139.30          87.80           10.90   
ma